In [36]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, cross_val_score, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

import time
import psutil
import threading
from memory_profiler import memory_usage

import joblib

# 1. Loading and preparing

In [37]:
df = pd.read_csv('D:/github/nids2/dataset/cleaned/cicids2017_cleaned.csv')

In [38]:
df.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Length of Fwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length',
       'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length',
       'Max Packet Length', 'Packet Length Mean', 'Packet Length Std',
       'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count',
       'ACK Flag Count', 'Average Packet Size', 'Subflow Fwd Bytes',
       'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_p

## 1.2. Preparing training and testing set

In [39]:
# splitting df for training and testing using stratified split
X = df.drop('Attack Type', axis=1) # features
y = df['Attack Type'] # target

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(X, y):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]

X_train = strat_train_set.drop("Attack Type", axis=1)
y_train = strat_train_set["Attack Type"]

X_test = strat_test_set.drop("Attack Type", axis=1)
y_test = strat_test_set["Attack Type"]

print(pd.DataFrame({
    "count (df)": df["Attack Type"].value_counts(),
    "count (train_set)": strat_train_set["Attack Type"].value_counts(),
    "count (test_set)": strat_test_set["Attack Type"].value_counts(),
    "proportion": strat_train_set["Attack Type"].value_counts(normalize=True),
})
)

                count (df)  count (train_set)  count (test_set)  proportion
Attack Type                                                                
Normal Traffic     2095057            1676045            419012    0.831124
DoS                 193745             154996             38749    0.076860
DDoS                128014             102411             25603    0.050784
Port Scanning        90694              72555             18139    0.035979
Brute Force           9150               7320              1830    0.003630
Web Attacks           2143               1714               429    0.000850
Bots                  1948               1559               389    0.000773


### Feature scaling

- Feature scaling is important to ensure all features contibute equally to the model's performance, especially for KNN that uses distance metrics. Based on analysis, the dataset contains outliers, hence the most suitable feature scaler is RobustScaler

In [40]:
rbscaler = RobustScaler()

# fit and transform training data, transform testing data
X_train_scaled = rbscaler.fit_transform(X_train)
X_test_scaled = rbscaler.transform(X_test)

### Attack Type resampling for training set

In [41]:
print(pd.DataFrame({
    "count": y_train.value_counts(),
    "proportion": y_train.value_counts(normalize=True)
})
)

                  count  proportion
Attack Type                        
Normal Traffic  1676045    0.831124
DoS              154996    0.076860
DDoS             102411    0.050784
Port Scanning     72555    0.035979
Brute Force        7320    0.003630
Web Attacks        1714    0.000850
Bots               1559    0.000773


- The training dataset shows a considerable imbalance with Normal Traffic as the majority. Hence, Normal Traffic is undersampled to reduce complexity

In [42]:
# Initializing the undersampling for the clean df
X_train_resampled, y_train_resampled = RandomUnderSampler(sampling_strategy={'Normal Traffic': 500000}, random_state=42).fit_resample(X_train, y_train)

# Initializing the undersampling for the scaled df
X_train_scaled, y_train_scaled = RandomUnderSampler(sampling_strategy={'Normal Traffic': 500000}, random_state=42).fit_resample(X_train_scaled, y_train)

In [43]:
print(pd.DataFrame({
    "count": y_train_resampled.value_counts(),
    "proportion": y_train_resampled.value_counts(normalize=True)
})
)

                 count  proportion
Attack Type                       
Normal Traffic  500000    0.594845
DoS             154996    0.184397
DDoS            102411    0.121837
Port Scanning    72555    0.086318
Brute Force       7320    0.008709
Web Attacks       1714    0.002039
Bots              1559    0.001855


- To further balance the training set, Synthetic Minority Over-Sampling (SMOTE) is used to oversample the minority classes

In [44]:
# Initializing the oversampling for the scaled df
X_train_resampled_scaled, y_train_resampled_scaled = SMOTE(sampling_strategy={'Bots': 2000, 'Web Attacks': 2000, 'Brute Force': 7500, 'Port Scanning': 73000, 'DDoS':102500, 'DoS': 200000}, random_state=42).fit_resample(X_train_scaled, y_train_scaled)

In [45]:
print(pd.DataFrame({
    "count": y_train_resampled_scaled.value_counts(),
    "proportion": y_train_resampled_scaled.value_counts(normalize=True)
})
)

del X_train_scaled, X_train, y_train, X, y, df


                 count  proportion
Attack Type                       
Normal Traffic  500000    0.563698
DoS             200000    0.225479
DDoS            102500    0.115558
Port Scanning    73000    0.082300
Brute Force       7500    0.008455
Bots              2000    0.002255
Web Attacks       2000    0.002255


# 2. Machine Learning Training

## 2.1. Random Forest

## 2.1.1. Hyperparameter Tuning

In [46]:
'''
# Defining the parameters for the Random Forest Classifier
param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
}

Creating the Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

# Saving results with the standard parameters
cv_sc_rf = cross_val_score(rf_model, X_train_resampled, y_train_resampled, cv=3, n_jobs=-1)
cv_sc_rf = np.mean(cv_sc_rf)

# Apply RandomizedSearchCV
random_search_rf = RandomizedSearchCV(estimator=rf_model, param_distributions=param_grid, n_iter=20, cv=3, n_jobs=-1, verbose=2)
random_search_rf.fit(X_train_resampled, y_train_resampled)

# Get the best parameters
print(f'Best Parameters: {random_search_rf.best_params_}')
print(f"Best Cross-Validation Score: {random_search_rf.best_score_}")
print(f"Cross-Validation from Standard: {cv_sc_rf}")

best_params_rf = random_search_rf.best_params_ if random_search_rf.best_score_ > cv_sc_rf else None

del random_search_rf
'''

'\nparam_grid = {\n    \'n_estimators\': [100, 150, 200],\n    \'max_depth\': [20, 30, None],\n    \'min_samples_split\': [2, 5, 10],\n    \'min_samples_leaf\': [1, 2, 4],\n    \'max_features\': [\'sqrt\', \'log2\'],\n}\n\nrf = RandomForestClassifier(random_state=42, n_jobs=-1)\n\nstart = time.time()\n# evaluate RF using 3-fold cross-validation; cv=3 means 3 filds, n_jobs=-1 means use all CPU cores in parallel\ncv_sc_rf = cross_val_score(rf, X_train_resampled, y_train_resampled, cv=3, n_jobs=-1, verbose=2)\ncv_sc_rf = np.mean(cv_sc_rf)\nprint(f"duration cv: {time.time()-start} ")\n\nrandom_search_rf = RandomizedSearchCV(estimator=rf, \n                                      param_distributions=param_grid, \n                                      n_iter=20, \n                                      cv=3, \n                                      n_jobs=-1, \n                                      verbose=2)\nstart = time.time()\nrandom_search_rf.fit(X_train_resampled, y_train_resampled)\nprint

In [ ]:
'''
Best Parameters: {'n_estimators': 150, 
                'min_samples_split': 2, 
                'min_samples_leaf': 2, 
                'max_features': 'sqrt', 
                'max_depth': 30}
Best Cross-Validation Score: 0.9986508913753411
Cross-Validation from Standard: 0.998389159543397
'''

## 2.1.2. Model fitting

In [ ]:
best_params_rf = {'n_estimators': 150, 
                'min_samples_split': 2, 
                'min_samples_leaf': 2, 
                'max_features': 'sqrt', 
                'max_depth': 30}
cv = 5
n_jobs = -1
random_state = 42

measurement_rf = {}

rf_model = RandomForestClassifier(**best_params_rf, random_state=random_state, n_jobs=n_jobs)

# Function to monitor CPU usage during training
cpu_usage = []
stop_flag = threading.Event()

def monitor_cpu():
    while not stop_flag.is_set():
        cpu_usage.append(psutil.cpu_percent(interval=0.1))

# Function to train the model
def train_model():
    rf_model.fit(X_train, y_train)

try:
    # Start CPU monitoring in a separate thread
    cpu_thread = threading.Thread(target=monitor_cpu)
    cpu_thread.start()

    # Measure memory usage and training time
    start_time = time.time()
    train_memory_rf = max(memory_usage((train_model,)))  # Measure peak memory usage
    training_time = time.time() - start_time

    # Stop CPU monitoring
    stop_flag.set()
    cpu_thread.join()

    # Add measurements
    measurement_rf['Memory Usage (MB)'] = train_memory_rf
    measurement_rf['Training Time (s)'] = training_time
    measurement_rf['Peak CPU Usage (%)'] = max(cpu_usage)
    measurement_rf['Average CPU Usage (%)'] = sum(cpu_usage) / len(cpu_usage) if cpu_usage else 0

    # Perform cross-validation
    cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=cv, n_jobs=n_jobs)

    return cv_scores_rf, measurement_rf, rf_model

except Exception as e:
    print(f"Error during Random Forest training: {e}")
    return None, None, None

# Making predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluating the model performance on the cross validation set vs accuracy on the test set
cv_scores_mean_rf = np.mean(cv_scores_rf)
print(f'Cross validation average score: {cv_scores_mean_rf:.4f} +/- standard deviation: {np.std(cv_scores_rf):.4f}')

accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f'Accuracy on the test set: {accuracy_rf:.4f}')

'''
Cross validation average score: 0.9987 +/- standard deviation: 0.0001
Accuracy on the test set: 0.9988
'''

## Model evaluation 

In [ ]:
## Making prediction
y_pred_rf = rf.predict(X_test)

cv_scores_mean_rf = np.mean(cv_scores_df)
print(f"Cross validation average score: {cv_scores_mean_rf:.4f}")

accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy on the test set: {accuracy_rf:.4f}")

### Confusion Matrix

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(10, 7))
sns.heatmap(cm_rf, annot=True, 
            fmt="d", 
            xticklabels=rf.classes_, 
            yticklabel = rf.classes_,
           cmap = 'Blues',
           cbar = False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Random Forest Confusion Matrix')
plt.show

### Classification report

In [ ]:
print(classification_report(y_test, y_pred_rf))

## Exporting model

In [ ]:
joblib.dump(rf, 'random_forest.joblib')

In [ ]:
# # Defining the parameter grid for XGBoost
# param_dist = {
#     'n_estimators': [100, 150, 200],
#     'max_depth': [3, 6, 9],
#     'learning_rate': [0.2, 0.3, 0.4],
#     'subsample': [0.7, 0.8, 1.0],
#     'colsample_bytree': [0.7, 0.8, 1.0],
#     'min_child_weight': [1, 5, 10],
# }

# # XGBoost's `multi:softmax` objective requires numerical labels for classification. Therefore, a mapping is necessary to convert categorical labels into numerical values before training the model.
# # # Creating the XGBoost Classifier
# xgb_model = xgb.XGBClassifier(objective='multi:softmax', num_class=len(y_train_resampled.unique()), random_state=42, n_jobs=-1)

# # Custom mapping for the attack types
# label_mapping = {
#     'Normal Traffic': 0,
#     'DoS': 1,
#     'DDoS': 2,
#     'Port Scanning': 3,
#     'Brute Force': 4,
#     'Web Attacks': 5,
#     'Bots': 6
# }
# y_train_resampled_mapped = y_train_resampled.map(label_mapping)
# y_test_mapped = y_test.map(label_mapping)

# # Saving results with the standard parameters
# cv_sc_xgb = cross_val_score(xgb_model, X_train_resampled, y_train_resampled_mapped, cv=3, n_jobs=-1)
# cv_sc_xgb = np.mean(cv_sc_xgb)

# # Perform RandomizedSearchCV
# random_search_xgb = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_dist, n_iter=30, cv=3, n_jobs=-1, verbose=2, random_state=42)
# random_search_xgb.fit(X_train_resampled, y_train_resampled_mapped)

# # Best parameters found by RandomizedSearchCV
# print(f'Best Parameters for XGBoost: {random_search_xgb.best_params_}')
# print(f"Best Cross-Validation Score: {random_search_xgb.best_score_}")
# print(f"Cross-Validation from Standard: {cv_sc_xgb}")

# best_params_xgb = random_search_xgb.best_params_ if random_search_xgb.best_score_ > cv_sc_xgb else None

# del random_search_xgb

# '''
# Best Parameters: 
# - `subsample`: 1.0
# - `n_estimators`: 150
# - `min_child_weight`: 1
# - `max_depth`: 3
# - `learning_rate`: 0.3
# - `colsample_bytree`: 0.7

# Best Parameters for XGBoost: {'subsample': 1.0, 'n_estimators': 100, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.2, 'colsample_bytree': 1.0}
# Best Cross-Validation Score: 0.9990827488980495
# Cross-Validation from Standard: 0.9990851282783

# '''

In [ ]:
# # Defining the parameters for KNN
# from sklearn.neighbors import KNeighborsClassifier

# param_grid_knn = {
#     'n_neighbors': [3, 5, 7],
#     'weights': ['uniform', 'distance'],
# }

# # Creating the KNN model
# knn_model = KNeighborsClassifier(n_jobs=-1)

# # Saving results with the standard parameters
# cv_sc_knn = cross_val_score(knn_model, X_train_resampled_scaled, y_train_resampled_scaled, cv=3, n_jobs=-1)
# cv_sc_knn = np.mean(cv_sc_knn)

# # Apply RandomizedSearchCV
# random_search_knn = RandomizedSearchCV(estimator=knn_model, param_distributions=param_grid_knn, n_iter=6, cv=3, n_jobs=-1, verbose=2)
# random_search_knn.fit(X_train_resampled_scaled, y_train_resampled_scaled)

# # Get the best parameters
# print(f'Best Parameters: {random_search_knn.best_params_}')
# print(f"Best Cross-Validation Score: {random_search_knn.best_score_}")
# print(f"Cross-Validation from Standard: {cv_sc_knn}")

# best_params_knn = random_search_knn.best_params_ if random_search_knn.best_score_ > cv_sc_knn else None

# del random_search_knn

# '''
# Best Parameters:

# - `weights`: distance
# - `n_neighbours`: 3
# '''